In [12]:
from pathlib import Path
import os
import pandas as pd


try:
    PROJECT_DIR = Path(__file__).resolve().parents[0]
except NameError:
    PROJECT_DIR = Path(os.getcwd()).resolve().parents[0]
    
DATA_PATH = PROJECT_DIR / "data" / "online_retail.csv"
OUTPUT_DIR = PROJECT_DIR / "outputs"

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

Removing Duplicates

In [13]:
df = pd.read_csv(DATA_PATH)
original_rows = len(df)
print(f"Original records: {original_rows}")

duplicates_count = int(df.duplicated().sum())
df = df.drop_duplicates().copy()
print(f"Number of duplicates removed: {duplicates_count}")
print(f"Records after removing duplicates: {len(df):,}")

Original records: 541909
Number of duplicates removed: 5268
Records after removing duplicates: 536,641


Convert data ypes

In [14]:
df["InvoiceDate"] = pd.to_datetime(df["InvoiceDate"],errors="coerce")
df["CustomerID"] = (df["CustomerID"].astype("Int64").astype("string"))
df["Description"] = df["Description"].fillna("Unkown product")
for column in ["InvoiceNo","StockCode","Country"]:
    df[column] = df[column].str.strip()

In [15]:
df["IsCancellation"] = df["InvoiceNo"].str.startswith("c", na=False)
df["IsNegativeQuantity"] = df["Quantity"] < 0
df["IsInvalidPrice"] = df["UnitPrice"] <= 0
df["IsMissingCustomer"] = df["CustomerID"].isna()
df["LineValue"] = df["Quantity"] * df["UnitPrice"]

df["IsValidSale"] = (
    ~df["IsCancellation"]
    & (df["Quantity"] >0)
    & (df["UnitPrice"] > 0)
    & df["InvoiceDate"].notna()
)
  
     
transactions_path = OUTPUT_DIR/"transactions_clean.csv"
df.to_csv(transactions_path,index=False) 

sales = df.loc[df["IsValidSale"]].copy()
sales_path = OUTPUT_DIR/"sales_data.csv"
sales.to_csv(sales_path,index=False)  

In [16]:
print("\n--- Cleaning summary ---")
print(f"Original rows: {original_rows:,}")
print(f"Duplicates removed: {duplicates_count:,}")
print(f"Rows after deduplication: {len(df):,}")
print(f"Valid sale lines: {len(sales):,}")
print(f"Rows excluded from sales analysis: {len(df) - len(sales):,}")

print("\n--- Missing values after cleaning ---")
print(df.isna().sum())

print("\n--- Sales dataset checks ---")
print(f"Non-positive quantities: {(sales["Quantity"] <= 0).sum()}")
print("Non-positive prices:", (sales["UnitPrice"] <= 0).sum())
print("Cancellation invoices:", sales["IsCancellation"].sum())

print("\n--- Sales date range ---")
print(sales["InvoiceDate"].min())
print(sales["InvoiceDate"].max())

print("\n--- Sales value ---")
print(f"Gross sales value: £{sales['LineValue'].sum():,.2f}")

print("\nFiles saved:")
print(transactions_path)
print(sales_path)


--- Cleaning summary ---
Original rows: 541,909
Duplicates removed: 5,268
Rows after deduplication: 536,641
Valid sale lines: 524,878
Rows excluded from sales analysis: 11,763

--- Missing values after cleaning ---
InvoiceNo                  0
StockCode                  0
Description                0
Quantity                   0
InvoiceDate                0
UnitPrice                  0
CustomerID            135037
Country                    0
IsCancellation             0
IsNegativeQuantity         0
IsInvalidPrice             0
IsMissingCustomer          0
LineValue                  0
IsValidSale                0
dtype: int64

--- Sales dataset checks ---
Non-positive quantities: 0
Non-positive prices: 0
Cancellation invoices: 0

--- Sales date range ---
2010-12-01 08:26:00
2011-12-09 12:50:00

--- Sales value ---
Gross sales value: £10,642,110.80

Files saved:
/home/FR3DDYN1GHT/Programming/projects/Future_Interns_task1_Online_retail_dataset/outputs/transactions_clean.csv
/home/FR3DDY